# 25 · Model tuning: HistGradientBoosting and Ridge

## Objective

Notebook 24 showed that an **untuned** HistGradientBoostingRegressor (HGB) beat the Ridge reference on all four metrics
in all five folds. This notebook asks one question: **can the default HGB, or Ridge, be meaningfully improved by a small,
predefined hyperparameter search?** It ends with **one** selected model configuration, ready for the final evaluation
on the untouched test set.

The Random Forest is **not** tuned. Its main weakness in notebook 24 was structural: it cannot predict above the prices
in its leaves, so expensive homes are under-predicted. The usual forest settings (number of trees, leaf size, number of
features per split) do not remove that ceiling. HGB was also better on every metric.

| Fixed | Value |
|---|---|
| Rows | the 16,219 training listings; the 4,081 test listings are removed from memory before anything else |
| Features | `build_model_features()` (notebook 23): 19 notebook-18 features + 6 locality-centre distances |
| Preprocessing | the notebook 23/24 `ColumnTransformer`, fitted inside each fold, **identical for every configuration** |
| Folds | `GroupKFold(n_splits=5, shuffle=True, random_state=42)`, `groups = property_group_id`, the same 5 folds for everything |
| Target | `log(price)` via `TransformedTargetRegressor`; metrics in rupees |
| Metrics | **MAE (primary)**, Median APE, RMSE, R²; mean and sample standard deviation over the 5 folds |
| Not used | the test set, locality target encoding, new features, early stopping |

## Part A. Setup

### A1. Load data, lock the test set, build the features

In [1]:
from pathlib import Path
import hashlib
import itertools
import sys
import time

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_CSV = PROJECT_ROOT / "data" / "processed" / "cleaned_listings.csv"
SPLIT_CSV = PROJECT_ROOT / "data" / "processed" / "train_test_split.csv"
LOCALITIES_CSV = PROJECT_ROOT / "data" / "extracted" / "localities.csv"
FEATURES_PY = PROJECT_ROOT / "src" / "features" / "build_features.py"

sys.path.insert(0, str(PROJECT_ROOT))
from src.features.build_features import DISTANCE_FEATURES, INPUT_COLUMNS, build_model_features


def file_md5(path):
    # A fingerprint of the file's bytes. If the file changes, the fingerprint changes.
    return hashlib.md5(path.read_bytes()).hexdigest()


# The same files as notebooks 23 and 24
EXPECTED_MD5 = {
    CLEANED_CSV: "3a81a385a9aca28d5b70322e1e143c08",
    SPLIT_CSV: "10693e86d1d8a55e606399d718a8f556",
    LOCALITIES_CSV: "8d9bb54dfc2fb49673e74b976ab73ace",
    FEATURES_PY: "4fb50b05aa072d1a9ae25851dbf1cc69",
}
for path, md5 in EXPECTED_MD5.items():
    assert file_md5(path) == md5, f"{path.name} is not the file used by notebooks 23-24"
    print(f"{path.name:28} {md5}")

cleaned_listings.csv         3a81a385a9aca28d5b70322e1e143c08
train_test_split.csv         10693e86d1d8a55e606399d718a8f556
localities.csv               8d9bb54dfc2fb49673e74b976ab73ace
build_features.py            4fb50b05aa072d1a9ae25851dbf1cc69


In [2]:
listings = pd.read_csv(CLEANED_CSV, low_memory=False)
split_assignment = pd.read_csv(SPLIT_CSV)
locality_table = pd.read_csv(LOCALITIES_CSV, usecols=["locality_id", "locality_name", "locality_lat", "locality_long"])

listings = listings.merge(split_assignment[["property_id", "split"]], on="property_id", validate="one_to_one")
assert len(listings) == 20300
assert (listings["split"] == "train").sum() == 16219
assert (listings["split"] == "test").sum() == 4081
train_groups = set(listings.loc[listings["split"] == "train", "property_group_id"])
test_groups = set(listings.loc[listings["split"] == "test", "property_group_id"])
assert train_groups.isdisjoint(test_groups), "A property group is on both sides!"

train = listings[listings["split"] == "train"].copy()
test_property_ids = set(listings.loc[listings["split"] == "test", "property_id"])
test_row_count = len(test_property_ids)
del listings   # the test rows are gone from memory from here on

model_features = build_model_features(train, locality_table)
y_train = train["price"]
groups = train["property_group_id"]

RANDOM_STATE = 42
cv_splits = list(GroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
                 .split(model_features, y_train, groups=groups))
expected_sizes = [(12997, 3222), (12980, 3239), (12961, 3258), (12989, 3230), (12949, 3270)]
for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
    assert (len(train_positions), len(validation_positions)) == expected_sizes[fold_number - 1]

print("Training rows:", len(train), "| test rows locked:", test_row_count)
print("Feature table:", model_features.shape, "| fold sizes identical to notebooks 19-24")

Training rows: 16219 | test rows locked: 4081
Feature table: (16219, 25) | fold sizes identical to notebooks 19-24


### A2. Preprocessing and fold loop (unchanged from notebook 24)

In [3]:
NUMERIC_FEATURES = ["bedrooms", "bathrooms", "log_area", "apartment_floor", "apartment_building_floors"] + DISTANCE_FEATURES
BINARY_FEATURES = ["area_missing", "area_unit_is_sqyd", "has_land_area_unit", "prime_location_flag", "in_named_project"]
CATEGORICAL_FEATURES = ["property_type", "balconies_band", "transaction_group", "furnishing", "possession_status",
                        "age_of_property", "facing_clean", "posted_by"]
LOCALITY_FEATURES = ["locality_code"]
assert sorted(NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES + LOCALITY_FEATURES) == sorted(model_features.columns)


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
             NUMERIC_FEATURES),
            ("binary", "passthrough", BINARY_FEATURES),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
            ("locality", OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist", sparse_output=False),
             LOCALITY_FEATURES),
        ]
    )


def make_log_price_model(regressor):
    return Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", TransformedTargetRegressor(regressor=regressor, func=np.log, inverse_func=np.exp)),
    ])


def median_ape(actual, predicted):
    return np.median(np.abs(predicted - actual) / actual) * 100


def positions_fingerprint(train_positions, validation_positions):
    # A short fingerprint of one fold's row positions, to prove every configuration used the same folds
    return hashlib.md5(train_positions.tobytes() + b"|" + validation_positions.tobytes()).hexdigest()


def run_cv(model):
    fold_rows = []
    out_of_fold = pd.Series(np.nan, index=model_features.index)
    fold_checks = []
    for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
        fold_model = clone(model)   # fresh, unfitted copy: preprocessing is re-learned from this fold's training part
        fold_model.fit(model_features.iloc[train_positions], y_train.iloc[train_positions])
        actual = y_train.iloc[validation_positions]
        predicted = fold_model.predict(model_features.iloc[validation_positions])
        out_of_fold.iloc[validation_positions] = predicted
        fold_rows.append({"fold": fold_number,
                          "MAE": mean_absolute_error(actual, predicted),
                          "RMSE": root_mean_squared_error(actual, predicted),
                          "R2": r2_score(actual, predicted),
                          "Median APE": median_ape(actual, predicted)})
        imputer = fold_model.named_steps["preprocess"].named_transformers_["numeric"].named_steps["impute"]
        fold_checks.append({
            "fingerprint": positions_fingerprint(train_positions, validation_positions),
            "imputer_medians": imputer.statistics_.copy(),
            "prediction_count": len(predicted),
        })
    return pd.DataFrame(fold_rows), out_of_fold, fold_checks


def summarise(fold_table):
    return {
        "MAE (₹ lakh)": fold_table["MAE"].mean() / 1e5,
        "MAE std": fold_table["MAE"].std() / 1e5,
        "Median APE (%)": fold_table["Median APE"].mean(),
        "Median APE std": fold_table["Median APE"].std(),
        "RMSE (₹ lakh)": fold_table["RMSE"].mean() / 1e5,
        "RMSE std": fold_table["RMSE"].std() / 1e5,
        "R²": fold_table["R2"].mean(),
        "R² std": fold_table["R2"].std(),
    }


# Every fold's expected imputer medians, computed once from each fold's training part
EXPECTED_MEDIANS = [model_features.iloc[train_positions][NUMERIC_FEATURES].median().values
                    for train_positions, _ in cv_splits]

### A3. The search plan, fixed before any model is evaluated

#### Ridge

`alpha` ∈ **[0.01, 0.1, 1, 10, 100]**: the list of notebook 20, covering four orders of magnitude. 5 configurations.

#### HistGradientBoosting

Every configuration keeps `early_stopping=False`, `loss="squared_error"` (on log price) and `random_state=42`.

| Parameter | Values | Default | Why these values |
|---|---|---|---|
| `learning_rate` & `max_iter` (as pairs) | (0.1, 200), (0.05, 400) | (0.1, 100) | the default stops after 100 small trees; both pairs give the model **more boosting** than the default, with the same "total learning" (rate × rounds = 20). The second pair uses smaller steps, which usually generalises a little better but takes twice as long |
| `max_leaf_nodes` | 31, 63 | 31 | size of each tree: 63 leaves can express more interactions (area × type × location) |
| `max_depth` | None, 8 | None | a depth limit stops a tree from growing one very long, narrow branch |
| `min_samples_leaf` | 10, 40 | 20 | smaller leaves can follow rare, expensive homes; larger leaves are smoother. The two values bracket the default |
| `l2_regularization` | 0, 1.0 | 0 | shrinks the leaf values, a second guard against over-fitting |

**2 × 2 × 2 × 2 × 2 = 32 configurations**, all evaluated on the same 5 folds, plus the untuned default from notebook 24
as the reference.

**Why learning rate and number of rounds are paired.** They work together: halving the rate needs roughly twice the
rounds to reach the same fit. Testing them as a full cross (2 × 2) would double the grid (and the largest fits take
about half a minute per fold) while adding mostly combinations that are known to be under- or over-trained. The pairs
keep the grid small and still test both parameters.

**Why no early stopping.** HGB's built-in early stopping holds back a *random* part of the training data, which is not
group-aware: copies of one property could be on both sides. The number of rounds is therefore part of the grid instead.

#### Selection rule (the same for both models)

1. For each configuration, compute the **mean CV MAE** over the 5 folds (the primary metric).
2. **Effectively tied** = mean MAE within **₹0.5 lakh** of the lowest mean MAE of that model family. ₹0.5 lakh is less
   than 1% of the MAE and about one-sixth of the fold-to-fold standard deviation (≈ ₹3 lakh), so smaller differences
   cannot be told apart from noise.
3. Among tied configurations, prefer the **simpler** one:
   - HGB: the smallest `max_iter × max_leaf_nodes` (the largest number of leaves the model can have); if still tied,
     the lowest mean MAE;
   - Ridge: the **largest** `alpha` (stronger shrinkage = a simpler model).
4. The test set plays no part, and no decision is based on a single fold. The fold-by-fold results of the selected
   configurations are reported (section 2).
5. **Final candidate:** the tuned configuration (Ridge or HGB) with the lowest mean CV MAE.

**One caveat, stated now:** choosing the best of 32 configurations on the same folds makes the winner's CV score
slightly **optimistic**. The honest estimate will come from the untouched test set.

In [4]:
RIDGE_ALPHAS = [0.01, 0.1, 1, 10, 100]

HGB_BOOSTING = [(0.1, 200), (0.05, 400)]   # (learning_rate, max_iter) pairs
HGB_MAX_LEAF_NODES = [31, 63]
HGB_MAX_DEPTH = [None, 8]
HGB_MIN_SAMPLES_LEAF = [10, 40]
HGB_L2 = [0.0, 1.0]

HGB_GRID = []
for (learning_rate, max_iter), leaves, depth, min_leaf, l2 in itertools.product(
        HGB_BOOSTING, HGB_MAX_LEAF_NODES, HGB_MAX_DEPTH, HGB_MIN_SAMPLES_LEAF, HGB_L2):
    HGB_GRID.append({"learning_rate": learning_rate, "max_iter": max_iter, "max_leaf_nodes": leaves,
                     "max_depth": depth, "min_samples_leaf": min_leaf, "l2_regularization": l2})

HGB_DEFAULT = {"learning_rate": 0.1, "max_iter": 100, "max_leaf_nodes": 31, "max_depth": None,
               "min_samples_leaf": 20, "l2_regularization": 0.0}
TIE_MARGIN_LAKH = 0.5

print("Ridge configurations:", len(RIDGE_ALPHAS))
print("HGB configurations:  ", len(HGB_GRID), "(+ the default reference)")


def make_hgb(settings):
    return make_log_price_model(HistGradientBoostingRegressor(**settings, early_stopping=False,
                                                              random_state=RANDOM_STATE))


def describe(settings):
    return (f"lr={settings['learning_rate']}, iter={settings['max_iter']}, leaves={settings['max_leaf_nodes']}, "
            f"depth={settings['max_depth']}, min_leaf={settings['min_samples_leaf']}, l2={settings['l2_regularization']}")

Ridge configurations: 5
HGB configurations:   32 (+ the default reference)


## Part B. Running the search

First the two untuned references (they must reproduce notebooks 23 and 24), then Ridge, then the HGB grid. Every fitted
fold's row positions and imputer medians are recorded for the checks in Part C.

In [5]:
all_checks = {}   # configuration name -> fold checks

ridge_reference_folds, ridge_reference_oof, all_checks["Ridge reference (alpha=1)"] = run_cv(
    make_log_price_model(Ridge(alpha=1.0)))
assert np.isclose(ridge_reference_folds["MAE"].mean() / 1e5, 69.365, atol=0.001)
assert np.isclose(ridge_reference_folds["Median APE"].mean(), 21.372, atol=0.001)
assert np.isclose(ridge_reference_folds["RMSE"].mean() / 1e5, 177.689, atol=0.001)
assert np.isclose(ridge_reference_folds["R2"].mean(), 0.491, atol=0.001)
print("Untuned Ridge reproduces notebook 23.")

hgb_default_folds, hgb_default_oof, all_checks["HGB default"] = run_cv(make_hgb(HGB_DEFAULT))
assert np.isclose(hgb_default_folds["MAE"].mean() / 1e5, 58.354, atol=0.001)
assert np.isclose(hgb_default_folds["Median APE"].mean(), 17.041, atol=0.001)
assert np.isclose(hgb_default_folds["RMSE"].mean() / 1e5, 164.759, atol=0.001)
assert np.isclose(hgb_default_folds["R2"].mean(), 0.561, atol=0.001)
assert np.allclose(hgb_default_folds["MAE"].values / 1e5, [55.363, 56.168, 62.269, 57.928, 60.040], atol=0.001)
print("Untuned HGB reproduces notebook 24 (means and every fold's MAE).")

Untuned Ridge reproduces notebook 23.


Untuned HGB reproduces notebook 24 (means and every fold's MAE).


In [6]:
ridge_results = []
ridge_fold_tables, ridge_oof = {}, {}
for alpha in RIDGE_ALPHAS:
    name = f"Ridge alpha={alpha}"
    fold_table, out_of_fold, all_checks[name] = run_cv(make_log_price_model(Ridge(alpha=alpha)))
    ridge_fold_tables[alpha], ridge_oof[alpha] = fold_table, out_of_fold
    ridge_results.append({"alpha": alpha, **summarise(fold_table)})
ridge_results = pd.DataFrame(ridge_results).set_index("alpha")
print("Ridge search done.")

Ridge search done.


In [7]:
hgb_results = []
hgb_fold_tables, hgb_oof = {}, {}
search_start = time.time()
for number, settings in enumerate(HGB_GRID, start=1):
    start = time.time()
    fold_table, out_of_fold, all_checks[f"HGB {number}"] = run_cv(make_hgb(settings))
    hgb_fold_tables[number], hgb_oof[number] = fold_table, out_of_fold
    hgb_results.append({"config": number, **settings, **summarise(fold_table), "seconds": time.time() - start})
hgb_results = pd.DataFrame(hgb_results).set_index("config")
hgb_results["max_depth"] = hgb_results["max_depth"].astype(object).where(hgb_results["max_depth"].notna(), "None")
print(f"HGB search: {len(HGB_GRID)} configurations x 5 folds in {(time.time() - search_start) / 60:.1f} minutes.")

HGB search: 32 configurations x 5 folds in 46.6 minutes.


## Part C. Validation checks

In [8]:
# 1. Identical folds for every configuration (and equal to a fresh GroupKFold with random_state=42)
fresh_fingerprints = [positions_fingerprint(train_positions, validation_positions)
                      for train_positions, validation_positions in
                      GroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(model_features, y_train, groups=groups)]
for name, checks in all_checks.items():
    assert [check["fingerprint"] for check in checks] == fresh_fingerprints, name
print(f"1. All {len(all_checks)} configurations used exactly the same 5 folds.")

# 2. No property group crosses a fold boundary
for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
    assert set(groups.iloc[train_positions]).isdisjoint(set(groups.iloc[validation_positions])), fold_number
print("2. No property_group_id on both sides of any fold.")

# 3. Test set never accessed
assert set(train["property_id"]).isdisjoint(test_property_ids)
assert (train["split"] == "train").all() and len(model_features) == 16219
print("3. None of the", test_row_count, "test listings was used; the test rows were deleted before any model was built.")

# 4. Preprocessing fitted only on each fold's training part
for name, checks in all_checks.items():
    for check, expected in zip(checks, EXPECTED_MEDIANS):
        assert np.allclose(check["imputer_medians"], expected), name
print("4. For every configuration and fold, the imputer medians = that fold's training-part medians.")

# 5. Valid predictions
validation_counts = np.zeros(len(model_features), dtype=int)
for train_positions, validation_positions in cv_splits:
    validation_counts[validation_positions] += 1
assert (validation_counts == 1).all()
all_out_of_fold = ([ridge_reference_oof, hgb_default_oof] + list(ridge_oof.values()) + list(hgb_oof.values()))
for predictions in all_out_of_fold:
    assert predictions.notna().all() and np.isfinite(predictions).all() and (predictions > 0).all()
    assert predictions.index.equals(model_features.index)
for name, checks in all_checks.items():
    for check, (train_positions, validation_positions) in zip(checks, cv_splits):
        assert check["prediction_count"] == len(validation_positions), name
print("5. Every configuration: one valid (finite, positive) prediction per validation row; each row validated once.")

# 6. References reproduced (asserted in Part B)
print("6. Untuned Ridge reproduces notebook 23 and untuned HGB reproduces notebook 24 (asserted in Part B).")

# 7. No target-derived or forbidden feature; source files unchanged
FORBIDDEN_COLUMNS = ["price", "price_band", "price_per_sqft", "loc_avg_price_sqft", "loc_avg_price_lmt",
                     "loc_price_quarter_prev", "loc_price_quarter_current", "loc_growth_quarterly_pct",
                     "loc_locality_rank", "locality_price_encoding", "latitude", "longitude",
                     "locality_centre_lat", "locality_centre_long"]
for column in FORBIDDEN_COLUMNS:
    assert column not in model_features.columns, column
for path, md5 in EXPECTED_MD5.items():
    assert file_md5(path) == md5, f"{path.name} changed!"
print("7. No target-derived feature, no target encoding, no coordinates; source files and build_features.py unchanged.")

1. All 39 configurations used exactly the same 5 folds.
2. No property_group_id on both sides of any fold.
3. None of the 4081 test listings was used; the test rows were deleted before any model was built.
4. For every configuration and fold, the imputer medians = that fold's training-part medians.
5. Every configuration: one valid (finite, positive) prediction per validation row; each row validated once.
6. Untuned Ridge reproduces notebook 23 and untuned HGB reproduces notebook 24 (asserted in Part B).


7. No target-derived feature, no target encoding, no coordinates; source files and build_features.py unchanged.


## 1. Tuning Results

### Ridge

In [9]:
ridge_results.round(3)

,MAE (₹ lakh),MAE std,Median APE (%),Median APE std,RMSE (₹ lakh),RMSE std,R²,R² std
alpha,,,,,,,,
0.01,69.365,3.182,21.334,0.463,177.479,20.601,0.493,0.029
0.10,69.364,3.183,21.344,0.460,177.499,20.614,0.492,0.029
1.00,69.365,3.199,21.372,0.430,177.689,20.734,0.491,0.029
10.00,69.653,3.320,21.506,0.372,179.284,21.559,0.482,0.030
100.00,71.508,3.423,22.182,0.392,185.439,23.392,0.447,0.032


### HistGradientBoosting: all 32 configurations, sorted by mean CV MAE

In [10]:
hgb_display = hgb_results.sort_values("MAE (₹ lakh)")
hgb_display[["learning_rate", "max_iter", "max_leaf_nodes", "max_depth", "min_samples_leaf", "l2_regularization",
             "MAE (₹ lakh)", "MAE std", "Median APE (%)", "RMSE (₹ lakh)", "R²", "seconds"]].round(3)

,learning_rate,max_iter,max_leaf_nodes,max_depth,min_samples_leaf,l2_regularization,MAE (₹ lakh),MAE std,Median APE (%),RMSE (₹ lakh),R²,seconds
config,,,,,,,,,,,,
27,0.05,400,63,None,40,0.0,57.275,2.626,16.398,161.385,0.577,169.251
28,0.05,400,63,None,40,1.0,57.277,2.543,16.223,161.781,0.576,178.377
32,0.05,400,63,8.0,40,1.0,57.344,2.772,16.428,161.902,0.575,72.760
24,0.05,400,31,8.0,40,1.0,57.410,2.777,16.480,162.029,0.574,81.634
31,0.05,400,63,8.0,40,0.0,57.427,2.751,16.380,161.798,0.576,91.824
20,0.05,400,31,None,40,1.0,57.438,2.754,16.528,163.266,0.568,115.535
23,0.05,400,31,8.0,40,0.0,57.453,2.827,16.633,162.073,0.574,82.163
19,0.05,400,31,None,40,0.0,57.477,2.811,16.583,162.936,0.570,100.082
4,0.10,200,31,None,40,1.0,57.528,2.740,16.508,161.873,0.576,74.765


### Which settings matter?

For each parameter value: the average of the mean CV MAE (and Median APE) over all 16 configurations that use it. A
large gap between the two values of a parameter means that parameter matters; a small gap means it hardly changes the
result.

In [11]:
effect_rows = []
hgb_results["boosting"] = ("lr=" + hgb_results["learning_rate"].astype(str) + ", iter=" + hgb_results["max_iter"].astype(str))
for parameter in ["boosting", "max_leaf_nodes", "max_depth", "min_samples_leaf", "l2_regularization"]:
    for value, part in hgb_results.groupby(parameter):
        effect_rows.append({"parameter": parameter, "value": value,
                            "average MAE (₹ lakh)": part["MAE (₹ lakh)"].mean(),
                            "average Median APE (%)": part["Median APE (%)"].mean(),
                            "average RMSE (₹ lakh)": part["RMSE (₹ lakh)"].mean()})
print("Default HGB for comparison: MAE", round(hgb_default_folds["MAE"].mean() / 1e5, 3),
      "| Median APE", round(hgb_default_folds["Median APE"].mean(), 3))
pd.DataFrame(effect_rows).set_index(["parameter", "value"]).round(3)

Default HGB for comparison: MAE 58.354 | Median APE 17.041


average MAE (₹ lakh)  average Median APE (%)  average RMSE (₹ lakh)
parameter         value                                                                                 
boosting          lr=0.05, iter=400                57.565                  16.511                163.079
                  lr=0.1, iter=200                 57.782                  16.608                163.001
max_leaf_nodes    31                               57.710                  16.631                163.174
                  63                               57.637                  16.488                162.907
max_depth         8.0                              57.659                  16.577                162.951
                  None                             57.688                  16.542                163.130
min_samples_leaf  10                               57.842                  16.600                164.054
                  40                               57.505                  16.519                162.027
l2_regularization 0.0                              57.685                  16.557                163.032
                  1.0                              57.662                  16.562                163.049

**Findings (tuning results):**

- **Ridge: `alpha` hardly matters between 0.01 and 10.** Mean CV MAE is ₹69.36–69.65 lakh for alphas 0.01–10 and
  gets clearly worse only at 100 (₹71.5 lakh). This is the same flat curve as notebook 20.
- **HGB: all 32 tuned configurations beat the untuned default** (₹58.35 lakh). They range from ₹57.28 lakh (config
  27) to ₹58.10 lakh (config 1). The spread is small: the best configuration is only 1.8% better than the default, and
  the whole grid spans less than ₹0.9 lakh, well below the fold-to-fold standard deviation (≈ ₹2.5–2.8 lakh).
- **The only setting with a clear effect is `min_samples_leaf`.** Leaves of at least 40 listings are better than
  leaves of 10 on average (MAE ₹57.51 vs ₹57.84 lakh, RMSE ₹162.0 vs ₹164.1 lakh). All 12 best configurations use 40.
  Larger leaves average more listings, so the predictions are less driven by a few unusual homes.
- The other settings change the average MAE by ₹0.2 lakh or less:
  - smaller steps with more rounds (lr 0.05 × 400) are ₹0.22 lakh better than lr 0.1 × 200, at twice the fitting time;
  - 63 leaves are ₹0.07 lakh better than 31;
  - the depth limit (8) and L2 regularisation (1.0) make practically no difference (≤ ₹0.03 lakh).
- **Conclusion:** the default HGB was already close to its best. Tuning gives a small, real gain, mainly from larger
  minimum leaf size and more boosting rounds, not a step change.

## 2. Best Configurations

The selection rule of A3, applied mechanically.

In [12]:
# Ridge: lowest mean MAE, then among those within the tie margin the LARGEST alpha
best_ridge_mae = ridge_results["MAE (₹ lakh)"].min()
ridge_tied = ridge_results[ridge_results["MAE (₹ lakh)"] <= best_ridge_mae + TIE_MARGIN_LAKH]
selected_alpha = ridge_tied.index.max()
print("Ridge: lowest mean MAE", round(best_ridge_mae, 3), "at alpha =", ridge_results["MAE (₹ lakh)"].idxmin())
print("       alphas within ₹0.5 lakh:", list(ridge_tied.index), "-> selected alpha =", selected_alpha)

# HGB: lowest mean MAE, then among those within the tie margin the smallest max_iter x max_leaf_nodes, then lowest MAE
best_hgb_mae = hgb_results["MAE (₹ lakh)"].min()
hgb_tied = hgb_results[hgb_results["MAE (₹ lakh)"] <= best_hgb_mae + TIE_MARGIN_LAKH].copy()
hgb_tied["size (max_iter x max_leaf_nodes)"] = hgb_tied["max_iter"] * hgb_tied["max_leaf_nodes"]
hgb_tied = hgb_tied.sort_values(["size (max_iter x max_leaf_nodes)", "MAE (₹ lakh)"])
selected_config = hgb_tied.index[0]
selected_hgb = HGB_GRID[selected_config - 1]
print()
print("HGB:   lowest mean MAE", round(best_hgb_mae, 3), "at config", hgb_results["MAE (₹ lakh)"].idxmin())
print("       configurations within ₹0.5 lakh:", len(hgb_tied))
print("       selected config", selected_config, ":", describe(selected_hgb))
hgb_tied[["learning_rate", "max_iter", "max_leaf_nodes", "max_depth", "min_samples_leaf", "l2_regularization",
          "size (max_iter x max_leaf_nodes)", "MAE (₹ lakh)", "Median APE (%)", "RMSE (₹ lakh)", "R²"]].round(3)

Ridge: lowest mean MAE 69.364 at alpha = 0.1
       alphas within ₹0.5 lakh: [0.01, 0.1, 1.0, 10.0] -> selected alpha = 10.0

HGB:   lowest mean MAE 57.275 at config 27
       configurations within ₹0.5 lakh: 22
       selected config 4 : lr=0.1, iter=200, leaves=31, depth=None, min_leaf=40, l2=1.0


,learning_rate,max_iter,max_leaf_nodes,max_depth,min_samples_leaf,l2_regularization,size (max_iter x max_leaf_nodes),MAE (₹ lakh),Median APE (%),RMSE (₹ lakh),R²
config,,,,,,,,,,,
4,0.10,200,31,None,40,1.0,6200,57.528,16.508,161.873,0.576
3,0.10,200,31,None,40,0.0,6200,57.596,16.675,161.918,0.575
8,0.10,200,31,8.0,40,1.0,6200,57.607,16.557,162.373,0.573
24,0.05,400,31,8.0,40,1.0,12400,57.410,16.480,162.029,0.574
20,0.05,400,31,None,40,1.0,12400,57.438,16.528,163.266,0.568
23,0.05,400,31,8.0,40,0.0,12400,57.453,16.633,162.073,0.574
19,0.05,400,31,None,40,0.0,12400,57.477,16.583,162.936,0.570
22,0.05,400,31,8.0,10,1.0,12400,57.686,16.690,163.588,0.567
17,0.05,400,31,None,10,0.0,12400,57.729,16.613,163.566,0.567


### Fold-by-fold metrics of the selected configurations and the untuned references

In [13]:
MODEL_TABLES = {
    "Ridge untuned (alpha=1)": ridge_reference_folds,
    f"Ridge tuned (alpha={selected_alpha})": ridge_fold_tables[selected_alpha],
    "HGB untuned (default)": hgb_default_folds,
    f"HGB tuned (config {selected_config})": hgb_fold_tables[selected_config],
}
MODEL_OOF = {
    "Ridge untuned (alpha=1)": ridge_reference_oof,
    f"Ridge tuned (alpha={selected_alpha})": ridge_oof[selected_alpha],
    "HGB untuned (default)": hgb_default_oof,
    f"HGB tuned (config {selected_config})": hgb_oof[selected_config],
}
for metric, scale, label in [("MAE", 1e5, "MAE (₹ lakh)"), ("Median APE", 1, "Median APE (%)"),
                             ("RMSE", 1e5, "RMSE (₹ lakh)"), ("R2", 1, "R²")]:
    table = pd.DataFrame({name: table[metric].values / scale for name, table in MODEL_TABLES.items()},
                         index=pd.Index(range(1, 6), name="fold"))
    print(label)
    print(table.round(3).to_string())
    print()

MAE (₹ lakh)
      Ridge untuned (alpha=1)  Ridge tuned (alpha=10.0)  HGB untuned (default)  HGB tuned (config 4)
fold                                                                                                
1                      66.365                    66.530                 55.363                53.875
2                      65.912                    66.147                 56.168                56.579
3                      72.874                    73.255                 62.269                60.921
4                      69.542                    69.666                 57.928                56.800
5                      72.134                    72.666                 60.040                59.464

Median APE (%)
      Ridge untuned (alpha=1)  Ridge tuned (alpha=10.0)  HGB untuned (default)  HGB tuned (config 4)
fold                                                                                                
1                      21.002                    21.141       

**Findings (best configurations):**

- **Ridge:** the lowest mean MAE is at alpha = 0.1 (₹69.364 lakh), but alphas 0.01, 0.1, 1 and 10 are all within ₹0.5
  lakh of it (they differ by at most ₹0.29 lakh). By the rule written down in advance, the **largest tied alpha, 10**,
  is selected. It is slightly *worse* than alpha = 1 in every fold (MAE +₹0.29 lakh on average). That is the price of the
  "prefer the simpler model" rule, and it is small. Ridge is not the final candidate in any case.
- **HGB:** the lowest mean MAE is config 27 (lr 0.05, 400 rounds, 63 leaves, min_samples_leaf 40: ₹57.275 lakh). 22
  configurations are within ₹0.5 lakh of it, so they count as tied. Among them, the rule prefers the smallest model
  (`max_iter × max_leaf_nodes` = 6,200). Three configurations have that size, and the one with the lowest MAE is
  **config 4: `learning_rate=0.1`, `max_iter=200`, `max_leaf_nodes=31`, `max_depth=None`, `min_samples_leaf=40`,
  `l2_regularization=1.0`**, at ₹57.528 lakh.
  - Config 27 is ₹0.25 lakh (0.4%) better but **four times larger** (25,200 vs 6,200) and about 2.3 times slower to fit.
  - The difference is well inside the tie margin and far below the fold-to-fold spread, so the simpler configuration
    is chosen.
- Config 4 **differs from the default in only three settings**: twice the rounds (200 instead of 100), larger minimum
  leaves (40 instead of 20) and a light L2 penalty (1.0 instead of 0).
- **Fold by fold**, tuned HGB (config 4) is better than the untuned default:
  - in 4 of 5 folds on MAE (fold 2 is ₹0.41 lakh worse);
  - in 5 of 5 on Median APE;
  - in 4 of 5 on RMSE and R².

  It beats the Ridge reference in every fold on every metric.
- Its Median APE varies more between folds than the default's (std 0.75 vs 0.42 points), mainly because fold 1 is
  especially good (15.2%). Folds 2–5 are between 16.7% and 17.0%.

## 3. Comparison With Untuned Models

In [14]:
comparison = pd.DataFrame([{"Model": name, **summarise(table)} for name, table in MODEL_TABLES.items()]).set_index("Model")
comparison.round(3)

,MAE (₹ lakh),MAE std,Median APE (%),Median APE std,RMSE (₹ lakh),RMSE std,R²,R² std
Model,,,,,,,,
Ridge untuned (alpha=1),69.365,3.199,21.372,0.430,177.689,20.734,0.491,0.029
Ridge tuned (alpha=10.0),69.653,3.320,21.506,0.372,179.284,21.559,0.482,0.030
HGB untuned (default),58.354,2.832,17.041,0.415,164.759,15.766,0.561,0.026
HGB tuned (config 4),57.528,2.740,16.508,0.754,161.873,14.770,0.576,0.035


In [15]:
def fold_wins(tuned, untuned):
    return {
        "MAE change (₹ lakh)": (tuned["MAE"].mean() - untuned["MAE"].mean()) / 1e5,
        "MAE change (%)": (tuned["MAE"].mean() / untuned["MAE"].mean() - 1) * 100,
        "Median APE change (points)": tuned["Median APE"].mean() - untuned["Median APE"].mean(),
        "RMSE change (₹ lakh)": (tuned["RMSE"].mean() - untuned["RMSE"].mean()) / 1e5,
        "R² change": tuned["R2"].mean() - untuned["R2"].mean(),
        "folds lower MAE": (tuned["MAE"] < untuned["MAE"]).sum(),
        "folds lower Median APE": (tuned["Median APE"] < untuned["Median APE"]).sum(),
        "folds lower RMSE": (tuned["RMSE"] < untuned["RMSE"]).sum(),
        "folds higher R²": (tuned["R2"] > untuned["R2"]).sum(),
    }


pd.DataFrame({
    "Ridge tuned vs untuned": fold_wins(ridge_fold_tables[selected_alpha], ridge_reference_folds),
    "HGB tuned vs untuned": fold_wins(hgb_fold_tables[selected_config], hgb_default_folds),
    "HGB tuned vs Ridge untuned": fold_wins(hgb_fold_tables[selected_config], ridge_reference_folds),
}).T.round(3)

,MAE change (₹ lakh),MAE change (%),Median APE change (points),RMSE change (₹ lakh),R² change,folds lower MAE,folds lower Median APE,folds lower RMSE,folds higher R²
Ridge tuned vs untuned,0.288,0.415,0.134,1.595,-0.009,0.0,1.0,0.0,0.0
HGB tuned vs untuned,-0.826,-1.416,-0.533,-2.886,0.014,4.0,5.0,4.0,4.0
HGB tuned vs Ridge untuned,-11.838,-17.066,-4.863,-15.816,0.084,5.0,5.0,5.0,5.0


**Findings (comparison with the untuned models):**

| | MAE (₹ lakh) | Median APE | RMSE (₹ lakh) | R² |
|---|---|---|---|---|
| Ridge untuned (alpha = 1) | 69.37 ± 3.20 | 21.37% | 177.69 | 0.491 |
| Ridge tuned (alpha = 10) | 69.65 ± 3.32 | 21.51% | 179.28 | 0.482 |
| HGB untuned (default) | 58.35 ± 2.83 | 17.04% | 164.76 | 0.561 |
| **HGB tuned (config 4)** | **57.53 ± 2.74** | **16.51%** | **161.87** | **0.576** |

- **Tuning Ridge brings nothing.** Its regularisation was never the limit; its linear form is.
- **Tuning HGB brings a small, consistent gain** over the default: MAE −₹0.83 lakh (−1.4%), Median APE −0.53 points, RMSE
  −₹2.9 lakh, R² +0.014. That is better on MAE in 4 of 5 folds and on Median APE in all 5.
  - The gain is larger than the ₹0.5 lakh tie margin, so the tuned configuration is preferred over the default.
  - It is modest, and part of it may be selection luck: the best of 32 configurations was picked on these same
    folds.
- **Tuned HGB vs the Ridge reference:** MAE −₹11.8 lakh (−17.1%), Median APE −4.9 points, RMSE −₹15.8 lakh, R² +0.08,
  better in all 5 folds on all four metrics.

## 4. Error Analysis

Out-of-fold predictions of the four models. For each group: MAE (₹ lakh), Median APE (%) and median signed % error
(+ = too high, − = too low). `price_band` is used **only for reporting**. The known difficult cases are ₹0–50 L,
₹5 Cr+, and houses/villas vs apartments.

In [16]:
RANGE_NAMES = ["0-50 L", "50-80 L", "80 L-1.2 Cr", "1.2-1.5 Cr", "1.5-1.75 Cr",
               "1.75-2 Cr", "2-3 Cr", "3-5 Cr", "5 Cr+"]
SHORT = dict(zip(MODEL_TABLES, ["Ridge", "Ridge tuned", "HGB", "HGB tuned"]))

errors = pd.DataFrame({
    "price": y_train,
    "price_band": train["price_band"],
    "type_family": np.where(train["property_type"].isin(["Residential House", "Villa"]), "house / villa",
                            "apartment-like"),
}, index=train.index)
for name, predictions in MODEL_OOF.items():
    errors[SHORT[name] + " abs"] = (predictions - y_train).abs()
    errors[SHORT[name] + " abs %"] = errors[SHORT[name] + " abs"] / y_train * 100
    errors[SHORT[name] + " signed %"] = (predictions - y_train) / y_train * 100


def compare_by(group_column, order, measure):
    grouped = errors.groupby(group_column)
    table = pd.DataFrame({"listings": grouped.size()})
    for name in MODEL_TABLES:
        if measure == "MAE (₹ lakh)":
            table[SHORT[name]] = grouped[SHORT[name] + " abs"].mean() / 1e5
        elif measure == "Median APE (%)":
            table[SHORT[name]] = grouped[SHORT[name] + " abs %"].median()
        else:
            table[SHORT[name]] = grouped[SHORT[name] + " signed %"].median()
    return table.reindex(order).round(1)


for measure in ["MAE (₹ lakh)", "Median APE (%)", "median signed error (%)"]:
    print("By price band:", measure)
    print(compare_by("price_band", RANGE_NAMES, measure).to_string())
    print()

By price band: MAE (₹ lakh)
             listings  Ridge  Ridge tuned    HGB  HGB tuned
price_band                                                 
0-50 L           1260   26.4         26.7   18.1       17.9
50-80 L          2291   20.8         20.7   18.4       18.3
80 L-1.2 Cr      2402   23.7         23.3   23.8       23.3
1.2-1.5 Cr       2287   34.3         33.8   29.6       29.4
1.5-1.75 Cr      1687   41.2         41.1   30.8       30.0
1.75-2 Cr        1616   49.2         49.0   38.1       38.4
2-3 Cr           2162   70.3         70.4   53.7       53.6
3-5 Cr           1550  130.7        131.9  105.6      105.4
5 Cr+             964  420.5        425.7  376.9      366.7

By price band: Median APE (%)


             listings  Ridge  Ridge tuned   HGB  HGB tuned
price_band                                                
0-50 L           1260   50.2         51.6  26.9       25.6
50-80 L          2291   19.9         20.1  16.5       16.9
80 L-1.2 Cr      2402   15.4         15.4  16.2       15.7
1.2-1.5 Cr       2287   17.5         17.1  14.8       13.8
1.5-1.75 Cr      1687   18.7         18.2  12.8       12.7
1.75-2 Cr        1616   19.0         19.5  13.6       13.8
2-3 Cr           2162   20.8         21.1  16.0       15.7
3-5 Cr           1550   29.1         29.0  22.8       21.9
5 Cr+             964   36.3         37.0  30.8       30.1

By price band: median signed error (%)
             listings  Ridge  Ridge tuned   HGB  HGB tuned
price_band                                                
0-50 L           1260   50.0         51.4  25.6       23.6
50-80 L          2291   16.0         16.9   8.0        7.7
80 L-1.2 Cr      2402    4.5          4.7   7.4        7.1
1.2-1.5 Cr      

In [17]:
for measure in ["MAE (₹ lakh)", "Median APE (%)", "median signed error (%)"]:
    print("Apartments vs houses/villas:", measure)
    print(compare_by("type_family", ["apartment-like", "house / villa"], measure).to_string())
    print()

Apartments vs houses/villas: MAE (₹ lakh)
                listings  Ridge  Ridge tuned   HGB  HGB tuned
type_family                                                  
apartment-like      9750   38.6         38.6  31.9       31.3
house / villa       6469  115.8        116.6  98.2       97.1

Apartments vs houses/villas: Median APE (%)


                listings  Ridge  Ridge tuned   HGB  HGB tuned
type_family                                                  
apartment-like      9750   17.5         17.5  13.6       13.4
house / villa       6469   30.2         30.2  24.3       23.6

Apartments vs houses/villas: median signed error (%)
                listings  Ridge  Ridge tuned  HGB  HGB tuned
type_family                                                 
apartment-like      9750    0.5          0.6  0.5        0.4
house / villa       6469    1.8          2.0  0.9        1.3



In [18]:
# The two known difficult ends in one table, plus the ₹20 Cr+ luxury homes
is_luxury = errors["price"] >= 20e7
rows = []
for name, predictions in MODEL_OOF.items():
    label = SHORT[name]
    rows.append({
        "Model": name,
        "₹0-50 L Median APE": errors.loc[errors["price_band"] == "0-50 L", label + " abs %"].median(),
        "₹0-50 L median signed %": errors.loc[errors["price_band"] == "0-50 L", label + " signed %"].median(),
        "₹5 Cr+ MAE (₹ lakh)": errors.loc[errors["price_band"] == "5 Cr+", label + " abs"].mean() / 1e5,
        "₹5 Cr+ median signed %": errors.loc[errors["price_band"] == "5 Cr+", label + " signed %"].median(),
        "₹20 Cr+ median signed %": errors.loc[is_luxury, label + " signed %"].median(),
        "within ±10% (%)": (errors[label + " abs %"] <= 10).mean() * 100,
        "within ±25% (%)": (errors[label + " abs %"] <= 25).mean() * 100,
    })
pd.DataFrame(rows).set_index("Model").round(1)

,₹0-50 L Median APE,₹0-50 L median signed %,₹5 Cr+ MAE (₹ lakh),₹5 Cr+ median signed %,₹20 Cr+ median signed %,within ±10% (%),within ±25% (%)
Model,,,,,,,
Ridge untuned (alpha=1),50.2,50.0,420.5,-34.5,-70.5,25.5,56.2
Ridge tuned (alpha=10.0),51.6,51.4,425.7,-35.2,-71.6,25.3,56.2
HGB untuned (default),26.9,25.6,376.9,-27.9,-64.1,32.0,65.3
HGB tuned (config 4),25.6,23.6,366.7,-25.9,-62.1,32.9,66.2


**Findings (error analysis):**

Tuning improves the known difficult cases **slightly**, and it does not make any of them worse. It does not change the
overall pattern:

| Difficult case | HGB untuned | HGB tuned | Ridge (reference) |
|---|---|---|---|
| ₹0–50 L: Median APE / median signed error | 26.9% / +25.6% | 25.6% / +23.6% | 50.2% / +50.0% |
| ₹5 Cr+: MAE / median signed error | ₹377 lakh / −27.9% | ₹367 lakh / −25.9% | ₹421 lakh / −34.5% |
| ₹20 Cr+ (48 listings): median signed error | −64.1% | −62.1% | −70.5% |
| apartment-like: Median APE | 13.6% | 13.4% | 17.5% |
| houses / villas: Median APE | 24.3% | 23.6% | 30.2% |

- The **pull toward the middle** is a little weaker after tuning at both ends, but it remains. Cheap homes are still
  over-predicted by about a quarter; ₹5 Cr+ homes are still under-predicted by about a quarter, and ₹20 Cr+ homes by
  about 60%.
- **Houses and villas** remain about **1.8 times** as hard as apartments (23.6% vs 13.4%).
- Some middle bands move very slightly the other way:
  - ₹50–80 L Median APE 16.5% → 16.9%;
  - ₹1.75–2 Cr 13.6% → 13.8%.

  These changes are small next to the gains elsewhere.
- Overall, 32.9% of listings are predicted within ±10% (untuned HGB 32.0%, Ridge 25.5%) and 66.2% within ±25%.

## 5. Tuning Decision

- **Ridge:** by the pre-set rule, alpha = 10. Alphas 0.01–10 are practically equivalent, and none improves on the
  untuned reference. Ridge stays in the project as the **simple, explainable reference**, not as the final model.
- **HistGradientBoosting:** tuning improved the default modestly but consistently (MAE −1.4%, Median APE −0.5 points,
  better in 4–5 of 5 folds on every metric). By the pre-set rule, **config 4** is selected: the smallest configuration
  among those tied with the best. The larger config 27 was not chosen, because its extra 0.4% MAE gain is within noise
  and it is four times the size.
- **Across the two families:** tuned HGB has by far the lower mean CV MAE (₹57.5 vs ₹69.7 lakh for tuned Ridge). It is
  also better on Median APE, RMSE and R² in every fold. By rule 5, it is the final candidate.
- No other model or configuration is carried forward. **No further tuning or model comparison is planned.**

## 6. Final Model Candidate

**`HistGradientBoostingRegressor`, learning log(price), with:**

| Setting | Value |
|---|---|
| `learning_rate` | 0.1 |
| `max_iter` | 200 |
| `max_leaf_nodes` | 31 |
| `max_depth` | None |
| `min_samples_leaf` | 40 |
| `l2_regularization` | 1.0 |
| `early_stopping` | False (its internal split is not group-aware) |
| `loss` | squared_error, on log(price) via `TransformedTargetRegressor(func=np.log, inverse_func=np.exp)` |
| `random_state` | 42 |
| Features | `build_model_features()`: 19 notebook-18 features + 6 locality-centre Haversine distances |
| Preprocessing | median imputation → `StandardScaler` (numeric); passthrough (binary); one-hot (categorical); one-hot with `min_frequency=10` (locality) |

**5-fold group-aware CV on the 16,219 training listings:** MAE ₹57.53 ± 2.74 lakh, Median APE 16.51 ± 0.75%, RMSE
₹161.87 ± 14.77 lakh, R² 0.576 ± 0.035. These numbers are **slightly optimistic**, because the configuration was chosen
on these folds. The test set will give the honest estimate.

In [19]:
final_candidate = {"model": "HistGradientBoostingRegressor", **selected_hgb, "early_stopping": False,
                   "random_state": RANDOM_STATE}
candidate_summary = summarise(hgb_fold_tables[selected_config])
print("Final model candidate (selected by the rule in A3):")
for key, value in final_candidate.items():
    print(f"  {key:20} {value}")
print()
print("Its 5-fold CV result on the training set (slightly optimistic, see A3):")
for key, value in candidate_summary.items():
    print(f"  {key:16} {value:.3f}")
# The candidate must be the lowest-MAE tuned configuration across both families, as the rule requires
assert candidate_summary["MAE (₹ lakh)"] < summarise(ridge_fold_tables[selected_alpha])["MAE (₹ lakh)"]

Final model candidate (selected by the rule in A3):
  model                HistGradientBoostingRegressor
  learning_rate        0.1
  max_iter             200
  max_leaf_nodes       31
  max_depth            None
  min_samples_leaf     40
  l2_regularization    1.0
  early_stopping       False
  random_state         42

Its 5-fold CV result on the training set (slightly optimistic, see A3):
  MAE (₹ lakh)     57.528
  MAE std          2.740
  Median APE (%)   16.508
  Median APE std   0.754
  RMSE (₹ lakh)    161.873
  RMSE std         14.770
  R²               0.576
  R² std           0.035


## 7. What Remains Before Final Test Evaluation

The final evaluation must be decided **completely before** the test set is opened, and run **once**:

1. **Refit** the final candidate pipeline (preprocessing + model, settings in section 6) on **all 16,219 training
   listings**. The preprocessing (medians, scaling, one-hot categories, rare-locality list) is learned from the whole
   training set. `build_model_features()` is applied to the test listings exactly as to the training listings.
2. **Predict the 4,081 test listings once.** Report:
   - MAE (**primary**), Median APE, RMSE, R²;
   - the same breakdowns as here: price band, property type, apartments vs houses/villas, ₹20 Cr+.
3. **Fix the comparison models now.** On the test set, show only the **Ridge reference** (alpha = 1, notebook 23) and the
   **Dummy median**, both refit on the full training set, as context. They are not candidates.
4. **No changes after seeing the test results.** No re-tuning, no feature changes, no removal of test listings (including
   probable entry errors such as the ≤ ₹2 lakh prices of notebook 20), and no switch to a different metric. If the test
   result is disappointing, report it as it is.
5. **Check the test set before predicting, without looking at prices:**
   - it still has 4,081 listings;
   - no `property_group_id` is shared with training;
   - the feature table has the same 25 columns.
6. **Report the limitations with the result:**
   - the target is **asking** prices, not sale prices;
   - the data is a non-random, price-range-driven sample (CLAUDE.md, point 8), so errors describe this collection, not
     the Hyderabad market;
   - `exp()` of a log prediction is closer to a median than a mean, so rupee predictions are slightly biased low;
   - ₹20 Cr+ homes, houses/villas and the cheapest listings are the weakest cases.
7. **Decide in advance what happens to the model after the test evaluation**, for example whether a deployment model is
   refit on training + test data. That refit would have no separate evaluation left and must be described as such.